# ScenarioRAG — 시나리오 "잠긴 종" 검색 (Chroma) + GM 진행 (이동·판정·flag·규칙 질문)

**흐름 한 줄**: 플레이어 입력 → `search_scenario(입력, 상태)` → 지금 볼 수 있는 시나리오 문서만 → 비슷한 순서로 k개

**상태(state)** 가 정하는 4가지
| 조건 | 문서 메타 | 통과하려면 |
|---|---|---|
| 언제 | `reveal_after` | 지금 장면 ≥ reveal_after |
| 누구에게 | `vis_all`, `vis_역할` | 전원 공개이거나, 내 직업 전용 |
| 선택의 결과 | `req_flag` | 조건 없음("")이거나, 내가 켠 flag |
| 어디에서 | `room` | 방과 무관("")이거나, 지금 있는 방 |

## ⚙️ 설정값

In [ ]:
# Ex00. 설정값
OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "bge-m3"

DATA = "/content/drive/MyDrive/KT 기본프로젝트"
SCN_ORIG_PATH = f"{DATA}/scenario/scenario.jsonl"
SCN_PATH = f"{DATA}/scenario/scenario_solo.jsonl"
ROOMS_PATH = f"{DATA}/scenario/rooms.yaml"
SCENES_ORIG_PATH = f"{DATA}/scenario/scenes.yaml"
SCENES_PATH = f"{DATA}/scenario/scenes_solo.yaml"
GOLD_PATH = f"{DATA}/scenario/golden_scenario.json"
COLLECTION = "scenario_drowned_bell"

ROLES = ["fighter", "rogue", "wizard", "cleric"]
ROLE = "fighter"
TOP_K = 4

## 0. 설치

In [ ]:
%%writefile /content/requirements_scn.txt
requests==2.32.4
chromadb
pyyaml
faiss-cpu==1.15.1

In [ ]:
# 설치
!pip install -r /content/requirements_scn.txt -q
!pip freeze | grep -iE "^(requests|chromadb|pyyaml|faiss-cpu)=="

## 1. Ollama 설치 · 서버 · 임베딩 모델 받기

In [ ]:
!apt-get install -y -qq zstd
!curl -fsSL http://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# Ex00-b. import · Ollama 서버 켜기
import os, re, json, time, subprocess
from collections import Counter, deque
import numpy as np, requests, yaml, chromadb
from tqdm.auto import tqdm
from google.colab import drive

def wait_server(url, seconds=120):
    for _ in range(seconds):
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print("서버 준비 완료:", url)
                return
        except Exception:
            pass
        time.sleep(1)
    print("서버가 아직 준비되지 않았습니다. ollama.log를 확인하세요")

os.environ["OLLAMA_KEEP_ALIVE"] = "1h"
subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)
wait_server(OLLAMA_URL)

In [ ]:
# 임베딩 모델 받기
!ollama pull {EMBED_MODEL}
!ollama list

## 2. 구글 드라이브

In [ ]:
drive.mount("/content/drive")

print(os.listdir(f"{DATA}/scenario"))

## 2-1. 싱글 플레이용으로 고치기

In [ ]:
# Ex00-c. 싱글 플레이용 시나리오 만들기
EDITS = {
    "scn-s5-monster-shadows": [("종실에 Shadow 3마리 (SRD p.319, CR 1/2, HP 27).",
                                "종실에 Shadow 3마리 (SRD p.319, CR 1/2, HP 27). 혼자인 주인공 앞에는 한 번에 한 마리씩만 물 위로 올라온다. "
                                "앞의 것이 쓰러지거나 멈춰야 다음 것이 올라온다.")],
    "scn-v8-zombie-monk": [("가 방 사이를 천천히 오간다.",
                            "가 방 사이를 천천히 오간다. 두 구는 서로 다른 방에 있어 한 번에 한 구씩 마주친다.")],
    "scn-v8-crab": [("Giant Crab 2마리", "Giant Crab 1마리"),
                    ("올라온 것들이다", "올라온 것이다"),
                    ("소리가 나면 몰려든다", "소리가 나면 다가온다")],
    "scn-s3-monster-ooze": [("전사가 앞장서면 갑옷이 먼저 노린다.", "금속 갑옷을 입었다면 갑옷부터 노린다.")],
    "scn-s5-secret-sluice": [("수문을 닫고 물을 빼려면 사람 둘이 양쪽 기어를 동시에 돌려야 한다.\n"
                              "근력(운동) DC 15를 두 명이 동시에 성공해야 하고, 물이 빠지는 데 10분이 걸린다.",
                              "수문을 닫고 물을 빼려면 양쪽 기어를 모두 돌려야 한다. 혼자라면 한쪽 기어를 쐐기나 밧줄로 고정해 두고 건너가 다른 쪽을 돌린다.\n"
                              "고정은 근력(운동) DC 12, 돌리기는 근력(운동) DC 15이고, 물이 빠지는 데 10분이 걸린다.")],
    "scn-g-sluice-desc": [("한 사람이 동시에 돌릴 수 없는 거리다.",
                           "한 사람이 동시에 돌릴 수 없는 거리라, 혼자라면 한쪽을 고정해 두고 건너가야 한다.")],
    "scn-q7-payoff-armor": [("일행 중 한 사람의 신호에 따라", "주인공의 신호에 따라")],
    "scn-q8-check-confront": [("\n증거는 먼저 본 사람이 말해 주어야 다른 사람이 쓸 수 있다.", "")],
    "scn-h-cleric-protection": [("한 사람에게 걸 수 있다", "자신에게 걸 수 있다")],
    "scn-s6-ending-toll": [("캠페인이 거기서 끝난다", "게임이 거기서 끝난다")],
    "scn-s5-secret-seal": [("마법사는 종의 정체를 간파한다. 지능(비전) DC 15.",
                            "종의 정체. 종 표면과 동심원 도식을 함께 살피면 지능(비전) 또는 지능(종교) DC 15로 알아챈다. "
                            "마법사는 이 판정에 이점을 받는다.")],
}
TO_ALL = ["scn-s5-secret-seal"]
SCENES_EDITS = [("players: 2~4인 핫시트", "players: 1인 (싱글 플레이)"),
                ("level: 1~3레벨 4인 파티 기준", "level: 1~3레벨 1인 기준"),
                ("cost: 누군가 종루에 남아 계속 종을 친다", "cost: 누군가(주인공, 할빅, 마을 사람) 종루에 남아 계속 종을 친다")]

def make_solo():
    rows = [json.loads(line) for line in open(SCN_ORIG_PATH, encoding="utf-8")]
    by = {r["id"]: r for r in rows}
    for did, pairs in EDITS.items():
        for old, new in pairs:
            n = by[did]["text"].count(old)
            assert n == 1, f"{did}: 원래 문장이 {n}번 있어요 → {old[:30]}"
            by[did]["text"] = by[did]["text"].replace(old, new)
    for did in TO_ALL:
        m = by[did]["meta"]
        m["vis_all"] = True
        for r in ROLES:
            m[f"vis_{r}"] = False
    for r in rows:
        r["text"] = r["text"].replace("일행", "주인공")
    with open(SCN_PATH, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    sc = open(SCENES_ORIG_PATH, encoding="utf-8").read()
    for old, new in SCENES_EDITS:
        assert sc.count(old) == 1, old
        sc = sc.replace(old, new)
    open(SCENES_PATH, "w", encoding="utf-8").write(sc)

    left = [r["id"] for r in rows if re.search(r"일행|두 명이 동시에|사람 둘이", r["text"])]
    print("고친 문서:", len(EDITS), "| 전원 공개로 바꾼 문서:", len(TO_ALL), "| 다인 표현 남은 문서:", left)

make_solo()

## 3. 시나리오 문서 읽기
문서가 짧아서(중앙값 157자) **문서 하나 = 검색 단위 하나**예요. 룰북처럼 다시 자르지 않아요.

In [ ]:
# Ex01. 시나리오 문서 · 방 지도 · 장면 설정 읽기
docs = [json.loads(line) for line in open(SCN_PATH, encoding="utf-8")]
rooms = yaml.safe_load(open(ROOMS_PATH, encoding="utf-8"))
scenes = yaml.safe_load(open(SCENES_PATH, encoding="utf-8"))
by_id = {d["id"]: d for d in docs}

## 4. 임베딩 · Chroma 컬렉션

In [ ]:
# Ex02. 임베딩 함수 (룰 RAG와 같은 bge-m3)
def embed_batch(texts, model=EMBED_MODEL):
    res = requests.post(f"{OLLAMA_URL}/api/embed",
                        json={"model": model, "input": texts, "options": {"num_ctx": 8192}},
                        timeout=300)
    res.raise_for_status()
    return res.json()["embeddings"]

In [ ]:
# Ex03. Chroma 컬렉션 만들기
client = chromadb.Client()
try:
    client.delete_collection(COLLECTION)
except Exception:
    pass
col = client.create_collection(COLLECTION, configuration={"hnsw": {"space": "cosine"}})

vecs = []
for i in tqdm(range(0, len(docs), 32)):
    vecs += embed_batch([d["text"] for d in docs[i:i + 32]])
col.add(ids=[d["id"] for d in docs],
        documents=[d["text"] for d in docs],
        embeddings=vecs,
        metadatas=[d["meta"] for d in docs])
print("컬렉션 문서 수:", col.count())

## 5. 상태 필터 + 검색
필터를 **검색보다 먼저** 걸어야 비밀이 새지 않아요. Chroma는 `where`로 거른 문서 안에서만 검색해요.
프롬프트로 "말하지 마"라고 하는 건 막는 장치가 아니에요.

In [ ]:
# Ex04. 상태 필터
def new_state(role=ROLE):
    return {"scene": 1, "role": role, "flags": set(), "room": rooms["start"]}

def where_of(state, use_room=True):
    conds = [
        {"reveal_after": {"$lte": state["scene"]}},
        {"$or": [{"vis_all": True}, {f"vis_{state['role']}": True}]},
        {"req_flag": {"$in": ["", *sorted(state["flags"])]}},
    ]
    if use_room:
        conds.append({"room": {"$in": ["", state["room"]]}})
    return {"$and": conds}

def visible(meta, state, use_room=True):
    m = meta
    return (m["reveal_after"] <= state["scene"]
            and (m["vis_all"] or m[f"vis_{state['role']}"])
            and (m["req_flag"] == "" or m["req_flag"] in state["flags"])
            and (not use_room or m["room"] in ("", state["room"])))

In [ ]:
# Ex05. 시나리오 검색
def search_scenario(query, state, k=TOP_K, use_room=True):
    res = col.query(query_embeddings=embed_batch([query]), n_results=k,
                    where=where_of(state, use_room),
                    include=["documents", "metadatas", "distances"])
    return [{"id": i, "score": 1 - dist, "kind": m["kind"], "text": t}
            for i, t, m, dist in zip(res["ids"][0], res["documents"][0],
                                     res["metadatas"][0], res["distances"][0])]

---
## 8. GM 진행 1단계: 방 이동 + 장면 묘사 (14B)
한 턴의 흐름
```
플레이어 입력 → 이동인가? (코드가 출구 이름과 비교)
   ├ 예 → 방 바꾸기 + 장면 번호 올리기 → 새 방 묘사
   └ 아니오 → search_scenario(입력, 상태) → GM 프롬프트 → 응답
응답 검사(깨짐·숫자 누설) → 걸리면 다시 생성
```


In [ ]:
# Ex09. 14B 받기 (백그라운드) + 호출 함수
GEN_MODEL = "qwen2.5:14b"
GEN_SEED = 42
GEN_CTX = 8192
HISTORY_TURNS = 6
GM_TRIES = 3

subprocess.Popen(f"nohup ollama pull {GEN_MODEL} > pull.log 2>&1 &", shell=True)

SYSTEM_GM = ("You are the game master of a Korean single-player D&D adventure. Write every reply in natural Korean (Hangul). "
             "Never use Chinese characters, Japanese, or any other language. "
             "Monster names may stay in English only when they appear that way in the reference.")

def _chat(prompt, system=SYSTEM_GM, temperature=0.7, seed=GEN_SEED, max_tokens=250):
    opts = {"num_ctx": GEN_CTX, "temperature": temperature, "num_predict": max_tokens}
    if seed is not None: opts["seed"] = seed
    res = requests.post(f"{OLLAMA_URL}/api/chat",
                        json={"model": GEN_MODEL, "stream": False, "options": opts,
                              "messages": [{"role": "system", "content": system},
                                           {"role": "user", "content": prompt}]},
                        timeout=600)
    res.raise_for_status()
    return res.json()["message"]["content"].strip()

In [ ]:
# Ex10. 방 이동
def exits_of(state):
    out = []
    for e in rooms["rooms"][state["room"]]["exits"]:
        if e.get("hidden") and e.get("requires") not in state["flags"]:
            continue
        out.append({"to": e["to"], "name": rooms["rooms"][e["to"]]["name"], "via": e["via"]})
    return out

def short(room_id):
    return rooms["rooms"][room_id]["name"].split(" (")[0]

def path_to(state, target):
    prev, q = {state["room"]: None}, deque([state["room"]])
    while q:
        x = q.popleft()
        for e in exits_of({**state, "room": x}):
            if e["to"] not in prev and (e["to"] in state["visited"] or e["to"] == target):
                prev[e["to"]] = x
                q.append(e["to"])
    if target not in prev:
        return None
    path, x = [], target
    while x != state["room"]:
        path.append(x)
        x = prev[x]
    return path[::-1]

def detect_move(text, state):
    ex = exits_of(state)
    m = re.match(r"\s*이동\s*(\d+)", text)
    if m and 1 <= int(m.group(1)) <= len(ex):
        return [ex[int(m.group(1)) - 1]["to"]]
    for e in ex:
        if short(e["to"]) in text or e["via"] in text:
            return [e["to"]]
    for rid in state["visited"] - {state["room"]}:
        if short(rid) in text:
            return path_to(state, rid)
    return None

def move(state, path):
    for room in path:
        state["room"] = room
        state["scene"] = max(state["scene"], rooms["rooms"][room]["scene"])
        state["visited"].add(room)

def new_game(role=ROLE):
    st = new_state(role)
    st["visited"] = {st["room"]}
    st["history"] = []
    return st

In [ ]:
# Ex11. GM 프롬프트
GM_TEMPLATE = """당신은 1인용 TRPG의 게임 마스터입니다. 아래 규칙을 지키세요.
1. [참고 자료]에 있는 내용만 사용해 지금 장면을 묘사하세요. 자료에 없는 인물, 장소, 사건, 물건을 지어내지 마세요.
   자료에 없는 관계(누구의 집인지, 누가 어디서 일하는지 등)도 만들지 마세요.
   자료에 적힌 지난 일(누가 사라졌다, 누가 어디로 갔다 등)은 지금 눈앞에서 일어나는 일로 묘사하지 마세요. 실종된 사람은 나타나지 않아요.
2. 플레이어에게 DC, HP, CR, SRD, 쪽수 같은 게임 수치와 출처는 말하지 마세요.
3. 플레이어를 '당신'이라고 부르세요. 당신의 행동, 생각, 감정, 의심을 대신 정하지 마세요. 당신이 한 행동의 결과와 보이는 것, 들리는 것만 묘사하세요. 당신의 대사도 지어내지 마세요.
4. NPC는 자료에 적힌 성격대로 짧게 말하게 하세요.
5. 2~4문장으로 쓰세요. 마지막에 "어떻게 하시겠어요?" 같은 질문은 쓰지 마세요.
6. 당신이 방금 한 행동의 결과가 불확실할 때만, 묘사 맨 끝 줄에 판정을 요청하세요: [판정] 능력(기술) DC 숫자
   판정을 요청할 때는 결과를 미리 묘사하지 말고, 시도하는 모습까지만 쓰세요. 다른 대상(숨은 괴물 등)에 대한 판정은 요청하지 마세요.
   능력(기술)은 이 중 하나만 쓰세요: 근력(운동), 민첩(은신), 민첩(손재주), 민첩(곡예), 지능(수사), 지능(역사), 지능(비전), 지능(종교), 지능(자연),
   지혜(감지), 지혜(통찰), 지혜(생존), 지혜(의학), 매력(설득), 매력(위협), 매력(기만), 도둑 도구

[현재 장소] {room}
[갈 수 있는 곳] {exits}

[참고 자료]
{docs}

[최근 대화]
{history}

[플레이어] {action}
[GM]"""

def fmt_history(state):
    h = state["history"][-HISTORY_TURNS:]
    return "\n".join(f"{who}: {text}" for who, text in h) or "(없음)"

def gm_prompt(action, state, hits, template=None):
    return (template or GM_TEMPLATE).format(
        room=rooms["rooms"][state["room"]]["name"],
        exits=", ".join(e["name"] for e in exits_of(state)) or "(없음)",
        docs="\n\n".join(f"- {h['text']}" for h in hits) or "(없음)",
        history=fmt_history(state), action=action)

In [ ]:
# Ex12. 응답 검사
BROKEN = re.compile(r"[^\uac00-\ud7a3\u3131-\u318e\x00-\x7f\s…·“”‘’—–]")
LEAK_NUM = re.compile(r"\b(HP|CR|SRD)\b|p\.\s*\d+|scn-")
CHECK = re.compile(r"\[판정\]\s*(.+?)\s*DC\s*(\d+)")

VALID_SKILLS = {"근력(운동)", "민첩(은신)", "민첩(손재주)", "민첩(곡예)", "지능(수사)", "지능(역사)", "지능(비전)", "지능(종교)",
                "지능(자연)", "지혜(감지)", "지혜(통찰)", "지혜(생존)", "지혜(의학)", "매력(설득)", "매력(위협)", "매력(기만)", "도둑 도구"}

CHECK_LINE = re.compile(r"\[판정\][^\n]*")

def cut_tail(text):
    t = text.rstrip()
    if re.search(r"[.!?…\"”’)]$", t):
        return t
    ends = list(re.finditer(r"[.!?…][\"”’)]?(?=\s|$)", t))
    return t[:ends[-1].end()] if ends else t

def split_check(text):
    found = CHECK_LINE.findall(text)
    body = cut_tail(CHECK_LINE.sub("", text).strip())
    m = CHECK.search(found[0]) if found else None
    if not m:
        return body, None
    skill = re.sub(r"\s+", "", m.group(1)).replace("도둑도구", "도둑 도구")
    return body, ({"skill": skill, "dc": int(m.group(2))} if skill in VALID_SKILLS else None)

def gm_problems(body):
    p = []
    if BROKEN.search(body): p.append("깨진 글자")
    if LEAK_NUM.search(body) or re.search(r"DC\s*\d+", body): p.append("수치 누설")
    if len(body) < 20: p.append("너무 짧음")
    if re.search(r"플레이어|주인공", body): p.append("호칭")
    return p

SKDC = re.compile(r"((?:근력|민첩|건강|지능|지혜|매력)\s*\(\s*[^)]+\)|도둑 도구)[^.\n]{0,20}?DC\s*(\d+)")

def check_candidates(hits):
    out = []
    for h in hits:
        head = h["text"].split("\n")[0][:40]
        for sent in re.split(r"(?<=[.다])\s+|\n", h["text"]):
            for m in SKDC.finditer(sent):
                skill = re.sub(r"\s+", "", m.group(1)).replace("도둑도구", "도둑 도구")
                item = (f"{head} … {sent.strip()}", skill, int(m.group(2)), h["id"])
                if skill in VALID_SKILLS and item not in out:
                    out.append(item)
    return out

PICK_TEMPLATE = """플레이어의 행동과 판정 후보입니다. 이 행동을 할 때 굴려야 하는 판정의 번호 하나만 쓰세요. 맞는 후보가 없으면 0을 쓰세요.

[행동] {action}

[후보]
{cands}

[번호]"""

def pick_check(action, hits, fallback):
    cands = check_candidates(hits)
    if not cands:
        return fallback
    lines = "\n".join(f"{i}) {s} → {sk} DC {dc}" for i, (s, sk, dc, _) in enumerate(cands, 1))
    out = _chat(PICK_TEMPLATE.format(action=action, cands=lines), temperature=0.0, max_tokens=5)
    m = re.search(r"\d+", out)
    n = int(m.group()) if m else 0
    if 1 <= n <= len(cands):
        s, sk, dc, did = cands[n - 1]
        return {"skill": sk, "dc": dc, "about": s, "doc": did}
    return fallback

In [ ]:
# Ex13. 한 턴 진행
def where_text(state):
    ex = exits_of(state)
    return "📍 " + rooms["rooms"][state["room"]]["name"] + " | 갈 수 있는 곳: " + \
           ", ".join(f"{i + 1}) {e['name']}" for i, e in enumerate(ex))

def narrate(action_for_gm, query, state, template=None, hits=None):
    hits = hits if hits is not None else search_scenario(query, state)
    prompt = gm_prompt(action_for_gm, state, hits, template)
    for t in range(GM_TRIES):
        body, check = split_check(_chat(prompt, seed=GEN_SEED + t))
        probs = gm_problems(body)
        if not probs:
            break
    if "깨진 글자" in probs:
        sents = re.split(r"(?<=[.!?…])\s+|\n+", body)
        body = " ".join(s for s in sents if s and not BROKEN.search(s)).strip() or "주변은 고요하다."
        probs = gm_problems(body)
    if "호칭" in probs:
        body = re.sub(r"주인공|플레이어", "당신", body)
        probs = gm_problems(body)
    return body, check, hits, probs

def gm_turn(action, state):
    path = detect_move(action, state)
    if path:
        move(state, path)
        query, action_for_gm = "이곳의 모습", f"{short(path[-1])}(으)로 들어간다"
    else:
        far = [rid for rid in rooms["rooms"] if rid != state["room"] and short(rid) in action and re.search(r"(가|간다|갈래|이동)", action)]
        if far:
            return {"text": f"{short(far[0])}(으)로는 여기서 바로 갈 수 없어요.\n\n" + where_text(state),
                    "hits": [], "problems": [], "check": None}
        query, action_for_gm = action, action
    body, check, hits, probs = narrate(action_for_gm, query, state)
    if path:
        check = None
    if check:
        check = pick_check(action, hits, check)
    state["history"] += [("플레이어", action), ("GM", body)]
    state["pending_check"] = {**check, "action": action} if check else None
    shown = body + (f"\n\n🎲 {check['skill']} 판정이 필요해요. (\"굴린다\"라고 입력)" if check else "")
    return {"text": shown + "\n\n" + where_text(state), "hits": [h["id"] for h in hits], "problems": probs, "check": check}

def play(action, dev=False):
    r = gm_turn(action, game)
    print(f"\n▶ {action}\n{r['text']}")
    if dev:
        print("   [개발] 장면", game["scene"], "| 방", game["room"], "| 근거", r["hits"], "| 검사", r["problems"])

In [ ]:
# 14B가 다 받아질 때까지 기다리기 (최대 20분)
t0 = time.time()
while time.time() - t0 < 1200:
    if GEN_MODEL in subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout:
        print(f"준비 완료 ({int(time.time() - t0)}초 기다림)")
        break
    print("받는 중...", int(time.time() - t0), "초")
    time.sleep(15)
else:
    print("20분이 지나도 안 받아졌어요. pull.log 끝부분:")
    print(open("pull.log").read()[-500:])

---
## 9. GM 진행 2단계: 주사위 판정 + flag
```
GM이 [판정] 요청 → 플레이어 "굴린다" → 코드가 d20 + 보정치 vs DC
   → 성공이면 flag 켜기 (숨은 길·조건부 문서가 열림) → 결과를 GM이 묘사
플레이어 입력에 사건 말(인형을 받는다 등)이 있으면 → 코드가 flag 켜기
```


In [ ]:
# Ex15. 캐릭터
PROF = 2
CHARS = {
    "fighter": {"mods": {"근력": 3, "민첩": 1, "건강": 2, "지능": -1, "지혜": 1, "매력": 0},
                "skills": {"운동", "감지", "생존", "위협"}},
    "rogue":   {"mods": {"근력": -1, "민첩": 3, "건강": 1, "지능": 1, "지혜": 1, "매력": 2},
                "skills": {"은신", "손재주", "감지", "수사", "설득", "도둑 도구"}},
    "wizard":  {"mods": {"근력": -1, "민첩": 2, "건강": 1, "지능": 3, "지혜": 1, "매력": 0},
                "skills": {"비전", "역사", "수사", "통찰"}},
    "cleric":  {"mods": {"근력": 1, "민첩": 0, "건강": 2, "지능": 0, "지혜": 3, "매력": 1},
                "skills": {"종교", "의학", "통찰", "설득"}},
}
ABIL = "근력|민첩|건강|지능|지혜|매력"
SKILL_ABIL = {"운동": "근력", "은신": "민첩", "손재주": "민첩", "곡예": "민첩", "도둑 도구": "민첩",
              "감지": "지혜", "생존": "지혜", "통찰": "지혜", "의학": "지혜",
              "수사": "지능", "역사": "지능", "비전": "지능", "종교": "지능", "자연": "지능",
              "설득": "매력", "위협": "매력", "기만": "매력"}

def parse_skill(text):
    m = re.search(rf"({ABIL})\s*(?:\(\s*([^)]+?)\s*\))?", text)
    if m:
        return m.group(1), m.group(2)
    for sk, ab in SKILL_ABIL.items():
        if sk in text:
            return ab, sk
    return None, None

def modifier(role, ability, skill):
    c = CHARS[role]
    return c["mods"][ability] + (PROF if skill in c["skills"] else 0)

In [ ]:
# Ex16. 주사위
import random
rng = random.Random()

def dc_from_docs(ability, skill, hit_ids, default):
    pat = re.escape(ability) + (rf"\s*\(\s*{re.escape(skill)}\s*\)" if skill else "") + r"\s*DC\s*(\d+)"
    for did in hit_ids:
        m = re.search(pat, by_id[did]["text"])
        if m:
            return int(m.group(1))
    return default

def roll_check(state, skill_text, dc):
    ability, skill = parse_skill(skill_text)
    if ability is None:
        ability, skill = "지혜", None
    d20 = rng.randint(1, 20)
    mod = modifier(state["role"], ability, skill)
    total = d20 + mod
    ok = d20 == 20 or (d20 != 1 and total >= dc)
    label = f"{ability}({skill})" if skill else ability
    return {"label": label, "ability": ability, "skill": skill, "d20": d20, "mod": mod, "total": total, "dc": dc, "ok": ok}

In [ ]:
# Ex17. flag 규칙
CHECK_FLAGS = [
    ("cloister_mid",        ["근력(운동)", "도둑 도구"],                 "opened_infirmary"),
    ("halvik_study",        ["도둑 도구", "민첩(손재주)", "매력(설득)"],  "opened_chest"),
    ("ledge_60",            ["지혜(생존)"],                              "tracked_rats"),
    ("scriptorium_desks",   ["지능(수사)"],                              "found_rota"),
    ("armor_corner",        ["지능(역사)"],                              "armor_calmed"),
    ("infirmary_inner",     ["지혜(감지)"],                              "found_quarry_passage"),
    ("scriptorium_shelves", ["지능(수사)"],                              "found_monk_diary"),
    ("bell_underside",      ["지혜(감지)", "근력(운동)"],                 "found_bell_relic"),
    ("bell_tongue",         ["근력(운동)", "민첩(은신)", "지혜(감지)"],   "found_last_pages"),
    ("belfry_platform",     ["매력(설득)"],                              "halvik_confessed"),
    ("well_side",           ["매력(설득)"],                              "heard_lullaby"),
]
EVENT_FLAGS = [
    ("village_square",  None, r"인형.*(받|챙|가져)|(받|챙|가져).*인형",           "got_rina_doll"),
    ("well_side",       None, r"(노래|자장가).*(듣|기다)",                         "heard_lullaby"),
    ("cloister_bodies", None, r"(도구|두루마리).*(챙|가져)",                       "took_garen_tools"),
    ("infirmary_inner", "opened_infirmary", r"귀마개.*(챙|가져)|(챙|가져).*귀마개", "got_wax_plugs"),
    ("belfry_platform", None, r"종을\s*(친|치|울)",                                "ending_toll"),
    ("belfry_platform", None, r"수문.*(복구|돌|닫)",                               "ending_drain"),
    ("belfry_platform", None, r"(포기|떠나|떠난|지상으로)",                         "ending_flee"),
]
COMBO_FLAGS = {"sera_pledge": ["got_rina_doll", "halvik_confessed"],
               "answered_the_call": ["found_last_pages", "halvik_confessed"],
               "armor_resolved": ["armor_calmed"]}
ENDINGS = {"ending_toll", "ending_drain", "ending_flee"}

def set_flag(state, flag):
    if flag in state["flags"]:
        return []
    state["flags"].add(flag)
    new = [flag]
    for f, need in COMBO_FLAGS.items():
        if f not in state["flags"] and all(x in state["flags"] for x in need):
            state["flags"].add(f)
            new.append(f)
    return new

def flags_from_check(state, roll):
    if not roll["ok"]:
        return []
    new = []
    for room, skills, flag in CHECK_FLAGS:
        if room == state["room"] and any(parse_skill(s) == (roll["ability"], roll["skill"]) for s in skills):
            new += set_flag(state, flag)
    return new

def flags_from_action(action, state):
    new = []
    for room, need, pat, flag in EVENT_FLAGS:
        if room == state["room"] and (need is None or need in state["flags"]) and re.search(pat, action):
            new += set_flag(state, flag)
    return new

In [ ]:
# Ex18. 판정 결과 묘사 + 한 턴 전체
RESULT_TEMPLATE = """당신은 TRPG 게임 마스터입니다. 아래 판정의 결과를 1~2문장으로 묘사하세요.
- [참고 자료]에 있는 내용만 쓰세요. 이름, 인물, 사건을 새로 만들지 마세요.
- 성공이면 [참고 자료]에서 이 행동으로 알아낼 만한 구체적인 사실 하나를 알려 주세요.
- 실패면 아무것도 알아내지 못했다고만 쓰세요. 참고 자료의 내용은 말하지 마세요.
- 수치(DC, 주사위 값)는 쓰지 마세요. 플레이어는 '당신'이라고 부르세요. [행동]에 나온 이름만 쓸 수 있고, 새 이름은 만들지 마세요.

[장소] {room}
[참고 자료]
{doc}

[행동] {action}
[판정 내용] {about}
[결과] {result}
[결과 묘사]"""

def narrate_result(pend, roll, state):
    about = pend.get("about", "").split(" … ")[-1] or roll["label"]
    doc = next((d["text"] for d in docs if d["id"] == pend.get("doc")), "(없음)") if roll["ok"] else "(없음)"
    prompt = RESULT_TEMPLATE.format(room=rooms["rooms"][state["room"]]["name"], doc=doc,
                                    action=pend["action"], about=about, result="성공" if roll["ok"] else "실패")
    for t in range(GM_TRIES):
        body, _ = split_check(_chat(prompt, seed=GEN_SEED + t, max_tokens=120))
        probs = gm_problems(body)
        if not probs:
            break
    return body, probs

def resolve_check(state):
    pend = state["pending_check"]
    hit_ids = [h["id"] for h in search_scenario(pend["action"], state, k=8)]
    dc = dc_from_docs(*parse_skill(pend["skill"]), hit_ids, pend["dc"])
    roll = roll_check(state, pend["skill"], dc)
    new = flags_from_check(state, roll)
    result = f"{pend['action']} → {roll['label']} 판정 {'성공' if roll['ok'] else '실패'}"
    if pend.get("about"):
        result += f" (판정 내용: {pend['about'].split(' … ')[-1]})"
    body, probs = narrate_result(pend, roll, state)
    hits = [{"id": pend["doc"]}] if pend.get("doc") else []
    state["history"] += [("플레이어", result), ("GM", body)]
    state["pending_check"] = None
    dice = f"🎲 {roll['label']}: d20 {roll['d20']} {roll['mod']:+d} = {roll['total']} → {'성공' if roll['ok'] else '실패'}"
    return {"text": dice + "\n\n" + body + "\n\n" + where_text(state), "hits": [h["id"] for h in hits],
            "problems": probs, "roll": roll, "flags_new": new}

def game_turn(action, state):
    if state.get("ended"):
        return {"text": "이야기가 끝났어요. game = new_game() 으로 새로 시작할 수 있어요.", "hits": [], "problems": []}
    if state.get("pending_check") and re.search(r"굴|주사위|판정", action):
        return resolve_check(state)
    new = flags_from_action(action, state)
    r = gm_turn(action, state)
    r["flags_new"] = new
    if new and state.get("pending_check"):
        state["pending_check"] = None
        r["text"] = re.sub(r"\n\n🎲[^\n]*", "", r["text"])
    if state["flags"] & ENDINGS:
        state["ended"] = True
        r["text"] += "\n\n🏁 결말에 도달했어요."
    return r

def play(action, dev=False):
    r = game_turn(action, game)
    print(f"\n▶ {action}\n{r['text']}")
    if dev:
        print("   [개발] 장면", game["scene"], "| 방", game["room"], "| 새 flag", r.get("flags_new"),
              "| 전체 flag", sorted(game["flags"]), "| 근거", r["hits"][:4], "| 검사", r["problems"])

In [ ]:
# 서버 확인 → 죽었으면 다시 켜기
try:
    requests.get(OLLAMA_URL, timeout=2)
    print("살아 있어요")
except Exception:
    subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)
    wait_server(OLLAMA_URL)
!ollama list

---
## 10. 입력 나누기: 규칙 질문은 룰 RAG로
```
입력 → 이동? (코드) → 판정 굴림? (코드) → 분류(14B): 규칙 / 진행
          규칙 → rule_rag.answer (RuleRAG_6 그대로)  → 📖 설명
          진행 → 시나리오 RAG + GM 묘사 (8~9번)
```
- 룰 RAG는 `rule_rag.py`로 불러와요. RuleRAG_6의 Ex00~Ex10-c를 그대로 옮긴 파일이라, 검색·답변 방식이 완성본과 같아요.
- 드라이브 `KT 기본프로젝트/` 폴더(룰 데이터가 있는 곳)에 `rule_rag.py`를 올려 두세요.
- 분류는 수업 [4] Ex06의 few-shot 방식이에요. `규칙:` 또는 `?`로 시작하면 분류 없이 바로 규칙으로 보내요.

In [ ]:
# Ex20. 룰 RAG 불러오기 (RuleRAG_6 완성본의 파이프라인)
import sys, importlib
if DATA not in sys.path:
    sys.path.insert(0, DATA)
import rule_rag as rr
importlib.reload(rr)

In [ ]:
# Ex21. 입력 분류
INTENT_TEMPLATE = """플레이어가 입력한 말이 어떤 종류인지 하나만 고르세요.
- 규칙: 룰북에 일반적으로 적힌 내용을 묻는 질문. 규칙, 판정 방법, 상태의 효과, 행동의 종류, 시야·환경 규칙,
        장비·물건의 가격과 성능, 마법 물건, 몬스터 정보처럼 게임 밖에서도 똑같이 답할 수 있는 것
- 진행: 지금 게임 속에서 하는 행동이나 대화, 또는 '여기·지금·이·저'처럼 이야기 속 장소·인물·물건·사건에 대해 묻는 말.
        지금 상황에서 어떤 행동을 해 보는 것("~하면 어떻게 돼?")도 진행이에요
'규칙' 또는 '진행' 한 단어만 쓰세요.

입력: 치명타가 나오면 피해를 어떻게 계산해?
답: 규칙
입력: 촌장에게 실종자에 대해 묻는다
답: 진행
입력: 밧줄 50피트는 얼마야?
답: 규칙
입력: 이 방에 숨겨진 게 있어?
답: 진행
입력: 투명해지는 마법 반지가 있어?
답: 규칙
입력: 저 갑옷은 왜 혼자 움직이는 거지?
답: 진행
입력: 은신하려면 어떤 판정을 해?
답: 규칙
입력: 여기서 횃불을 끄면 어떻게 돼?
답: 진행

입력: {text}
답:"""

def classify(text):
    if re.match(r"\s*(규칙|룰)\s*[:：]", text) or text.strip().startswith("?"):
        return "규칙"
    out = _chat(INTENT_TEMPLATE.format(text=text), temperature=0.0, max_tokens=5)
    return "규칙" if "규칙" in out else "진행"

In [ ]:
# Ex23. 한 턴 전체 (최종)
def rule_turn(text):
    q = re.sub(r"^\s*(규칙|룰)\s*[:：]\s*|^\s*\?\s*", "", text)
    a = rr.answer(q, attempts=rr.CHAT_ATTEMPTS, show_orig=False)
    if a["status"] == "ok":
        a["explain"] = re.sub(r"\s*\([A-Za-z][A-Za-z \-']*\)", "", a["explain"])
        return "📖 " + a["explain"], a
    return "📖 이 질문에 맞는 규칙을 확실히 찾지 못했어요. 질문을 조금 바꿔서 다시 물어봐 주세요.", a

def game_turn(action, state):
    if state.get("ended"):
        return {"text": "이야기가 끝났어요. game = new_game() 으로 새로 시작할 수 있어요.", "hits": [], "problems": []}
    if state.get("pending_check") and re.search(r"굴|주사위|판정", action):
        return resolve_check(state)
    if not detect_move(action, state) and classify(action) == "규칙":
        text, a = rule_turn(action)
        forced = re.match(r"\s*(규칙|룰)\s*[:：]|\s*\?", action)
        if a["status"] == "ok" or forced:
            return {"text": text, "hits": [a.get("title")], "problems": a.get("warn", []), "kind": "규칙"}
    new = flags_from_action(action, state)
    r = gm_turn(action, state)
    r["flags_new"], r["kind"] = new, "진행"
    if state["flags"] & ENDINGS:
        state["ended"] = True
        r["text"] += "\n\n🏁 결말에 도달했어요."
    return r

def play(action, dev=False):
    r = game_turn(action, game)
    print(f"\n▶ {action}\n{r['text']}")
    if dev:
        print("   [개발]", r.get("kind", ""), "| 장면", game["scene"], "| 방", game["room"], "| 새 flag", r.get("flags_new"),
              "| 근거", (r["hits"] or [])[:4], "| 검사", r["problems"])

## 11. 전투 (전부 코드, 묘사만 14B)

In [ ]:
# Ex25. 몬스터 수치
def roll_dice(expr):
    expr = re.sub(r"\s+", "", expr).replace("−", "-")
    total = 0
    for sign, n, d, k in re.findall(r"([+-]?)(?:(\d*)d(\d+)|(\d+))", expr):
        v = sum(rng.randint(1, int(d)) for _ in range(int(n or 1))) if d else int(k)
        total += -v if sign == "-" else v
    return total

def srd_monster(name):
    c = next((c for c in rr.chunks if c["meta"]["title"].lower() == name.lower()), None)
    if c is None:
        return None
    t = re.sub(r"\s+", " ", c["text"]).replace("−", "-")
    ac = re.search(r"\bAC\s*(\d+)", t)
    hp = re.search(r"\bHP\s*(\d+)", t)
    atk = re.search(r"(?:Melee|Ranged) Attack Roll:\s*\+(\d+).*?Hit:\s*\d+\s*\(([^)]+)\)\s*([A-Za-z]+)", t, re.S)
    ini = re.search(r"Initiative\s*([+-]\d+)", t)
    dex = re.search(r"\bDex\w*\W*?(\d+)\s*([+-]\d+)", t, re.I)
    if not (ac and hp and atk):
        return None
    return {"name": name, "ac": int(ac.group(1)), "hp": int(hp.group(1)), "atk": int(atk.group(1)),
            "dmg": re.sub(r"\s+", "", atk.group(2)), "dmg_type": atk.group(3), "init": int(ini.group(1)) if ini else (int(dex.group(2)) if dex else 0)}

titles = sorted({c["meta"]["title"] for c in rr.chunks}, key=len, reverse=True)
SCN_MONSTERS = {}
for d in docs:
    if d["meta"]["kind"] != "monster":
        continue
    found = []
    for t in titles:
        if re.search(rf"\b{re.escape(t)}\b", d["text"]) and not any(t in f for f, _ in found):
            stat = srd_monster(t)
            if stat:
                m = re.search(rf"{re.escape(t)}\s*(\d+)\s*마리", d["text"])
                found.append((t, int(m.group(1)) if m else 1))
    SCN_MONSTERS[d["id"]] = found

MONSTER_KO = {
    "Swarm of Rats": "쥐 떼", "Shrieker Fungus": "비명 버섯", "Gray Ooze": "회색 점액괴물",
    "Animated Armor": "움직이는 갑옷", "Shadow": "그림자", "Specter": "망령",
    "Giant Crab": "거대 게", "Zombie": "좀비", "Giant Octopus": "거대 문어",
}

def ko_monster(text):
    for en in sorted(MONSTER_KO, key=len, reverse=True):
        text = re.sub(rf"(?<![A-Za-z]){re.escape(en)}(?![A-Za-z])", MONSTER_KO[en], text)
    return text

In [ ]:
# Ex26. 플레이어 전투 수치 (1레벨 기본값
COMBAT_STATS = {
    "fighter": {"ac": 16, "hp": 12, "atk": 5, "dmg": "1d8+3", "weapon": "장검"},
    "rogue":   {"ac": 14, "hp": 9,  "atk": 5, "dmg": "1d8+3", "weapon": "레이피어"},
    "wizard":  {"ac": 12, "hp": 7,  "atk": 5, "dmg": "1d10",  "weapon": "화염 화살"},
    "cleric":  {"ac": 16, "hp": 10, "atk": 3, "dmg": "1d6+1", "weapon": "철퇴"},
}
MENU = "1) 공격  2) 회피  3) 도주"

def foe_docs_here(state):
    return [d["id"] for d in docs
            if d["id"] in SCN_MONSTERS and SCN_MONSTERS[d["id"]] and visible(d["meta"], state)
            and d["meta"]["room"] == state["room"] and d["id"] not in state.setdefault("defeated", set())]

def start_combat(state):
    here = foe_docs_here(state)
    if not here:
        return {"text": "여기에는 싸울 상대가 없어요.", "hits": [], "problems": []}
    did = here[0]
    foes = []
    for name, k in SCN_MONSTERS[did]:
        s = srd_monster(name)
        foes += [{**s, "max": s["hp"], "label": f"{MONSTER_KO.get(name, name)}{' ' + str(i + 1) if k > 1 else ''}"} for i in range(k)]
    me = COMBAT_STATS[state["role"]]
    state.setdefault("hp", me["hp"])
    my_init = rng.randint(1, 20) + CHARS[state["role"]]["mods"]["민첩"]
    foe_init = rng.randint(1, 20) + foes[0]["init"]
    state["combat"] = {"doc": did, "foes": foes, "round": 1}
    log = [f"⚔️ 전투 시작! 선제권: 당신 {my_init} / {foes[0]['label']} {foe_init}"]
    if foe_init > my_init:
        log += foe_attack(state, dodge=False)
    return {"text": "\n".join(log) + combat_status(state), "hits": [did], "problems": []}

In [ ]:
# Ex27. 전투 라운드
def combat_status(state):
    c = state.get("combat")
    if not c:
        return ""
    foe = next(f for f in c["foes"] if f["hp"] > 0)
    left = sum(f["hp"] > 0 for f in c["foes"])
    return f"\n\n❤️ 당신 HP {state['hp']}/{COMBAT_STATS[state['role']]['hp']} | 👹 {foe['label']} HP {foe['hp']}/{foe['max']}" + \
           (f" (남은 적 {left})" if left > 1 else "") + f"\n{MENU}"

def foe_attack(state, dodge):
    foe = next(f for f in state["combat"]["foes"] if f["hp"] > 0)
    d20 = min(rng.randint(1, 20), rng.randint(1, 20)) if dodge else rng.randint(1, 20)
    total = d20 + foe["atk"]
    if d20 == 20 or (d20 != 1 and total >= COMBAT_STATS[state["role"]]["ac"]):
        dmg = max(1, roll_dice(foe["dmg"]) + (roll_dice(re.sub(r"[+-]\d+$", "", foe["dmg"])) if d20 == 20 else 0))
        state["hp"] = max(0, state["hp"] - dmg)
        return [f"👹 {foe['label']}의 공격: {total} → 명중! 피해 {dmg}"]
    return [f"👹 {foe['label']}의 공격: {total} → 빗나감"]

def player_attack(state):
    me = COMBAT_STATS[state["role"]]
    foe = next(f for f in state["combat"]["foes"] if f["hp"] > 0)
    d20 = rng.randint(1, 20)
    total = d20 + me["atk"]
    if d20 == 20 or (d20 != 1 and total >= foe["ac"]):
        dmg = max(1, roll_dice(me["dmg"]) + (roll_dice(re.sub(r"[+-]\d+$", "", me["dmg"])) if d20 == 20 else 0))
        foe["hp"] = max(0, foe["hp"] - dmg)
        return [f"🗡️ 당신의 {me['weapon']}: {total} → 명중! 피해 {dmg}" + (" (치명타)" if d20 == 20 else "")]
    return [f"🗡️ 당신의 {me['weapon']}: {total} → 빗나감"]

COMBAT_TEMPLATE = """당신은 TRPG 게임 마스터입니다. 아래 전투 기록을 2문장 이내로 생생하게 묘사하세요.
- 기록에 있는 결과(명중·빗나감·쓰러짐)만 쓰세요. 숫자는 쓰지 마세요. 플레이어는 '당신'이라고 부르세요.
- 몬스터 이름은 기록에 적힌 한국어 이름을 그대로 쓰세요. 영어로 바꾸지 마세요.
- [직전 묘사]와 같은 표현(같은 첫마디, 같은 꾸밈말)을 다시 쓰지 마세요. 공격 방향, 몸놀림, 소리, 주변 물 등 다른 부분을 묘사하세요.

[직전 묘사]
{before}

[전투 기록]
{log}

[묘사]"""

def ig(word):
    ch = word[-1]
    return ("이" if (ord(ch) - 0xAC00) % 28 else "가") if "가" <= ch <= "힣" else "이(가)"

def combat_turn(action, state):
    c = state["combat"]
    a = action.strip()
    choice = next((i for i, w in ((1, "공격"), (2, "회피"), (3, "도주")) if a.startswith(str(i)) or w in a), None)
    if choice is None:
        return {"text": f"전투 중이에요. 번호로 골라 주세요.\n{MENU}", "hits": [], "problems": []}
    log = []
    if choice == 3:
        log += foe_attack(state, dodge=False) + ["당신은 전투에서 빠져나왔어요."]
        state["combat"] = None
    else:
        if choice == 1:
            log += player_attack(state)
        for f in c["foes"]:
            if f["hp"] == 0 and not f.get("down"):
                f["down"] = True
                log.append(f"{f['label']}{ig(f['label'])} 쓰러졌어요.")
        if all(f["hp"] == 0 for f in c["foes"]):
            log.append("모든 적이 쓰러졌어요.")
            state["defeated"].add(c["doc"])
            state["combat"] = None
        else:
            log += foe_attack(state, dodge=(choice == 2))
            c["round"] += 1
    if state["hp"] == 0:
        log.append("💀 당신은 쓰러졌습니다. 이야기가 여기서 끝납니다.")
        state["ended"], state["combat"] = True, None
    told = c.setdefault("told", [])
    story = _chat(COMBAT_TEMPLATE.format(before="\n".join(told[-2:]) or "(없음)", log="\n".join(log)),
                  seed=GEN_SEED + c["round"], max_tokens=120)
    story = story if not gm_problems(story) else ""
    if story:
        told.append(story)
    state["history"] += [("플레이어", "전투: " + ["공격", "회피", "도주"][choice - 1]), ("GM", story or " / ".join(log))]
    return {"text": "\n".join(log) + ("\n\n" + story if story else "") + combat_status(state), "hits": [c["doc"]], "problems": []}

In [ ]:
# Ex28. game_turn에 전투 붙이기 (Ex23의 game_turn을 감싸요)
if not getattr(game_turn, "has_combat", False):
    game_turn_base = game_turn

def game_turn(action, state):
    if state.get("combat"):
        return combat_turn(action, state)
    if action.strip() in ("/전투", "전투"):
        return start_combat(state)
    r = game_turn_base(action, state)
    if not state.get("ended") and foe_docs_here(state):
        r["text"] += "\n\n⚔️ 이곳에는 적이 있어요. '/전투'로 싸움을 시작할 수 있어요."
    return r
game_turn.has_combat = True

## 12. 긴 기억 (기억 RAG)
- 매 턴 "플레이어: … / GM: …"를 그대로 memory 컬렉션에 저장 (턴 번호·방·장면 메타데이터)
- 다음 턴에 입력과 비슷한 지난 기억 3개를 찾아 GM 프롬프트 [이전 기억]에 넣어요 (최근 대화에 이미 있는 턴은 빼요)
- 요약하지 않아요. 요약하면 이름(할빅 등)이 빠져서 나중에 못 찾아요.

In [ ]:
# Ex30. 기억 컬렉션 + 저장 · 검색
MEM_K = 3
try:
    client.delete_collection("memory")
except Exception:
    pass
mem = client.create_collection("memory", configuration={"hnsw": {"space": "cosine"}})

def remember(state, action, body):
    state["turn"] = state.get("turn", 0) + 1
    text = f"[{state['turn']}턴 · {short(state['room'])}] 플레이어: {action} / GM: {body}"
    mem.add(ids=[f"t{state['turn']}"], documents=[text], embeddings=embed_batch([text]),
            metadatas=[{"turn": state["turn"], "scene": state["scene"], "room": state["room"]}])

def recall(query, state, k=MEM_K):
    cutoff = state.get("turn", 0) - HISTORY_TURNS // 2 + 1
    if cutoff <= 1 or mem.count() == 0:
        return []
    res = mem.query(query_embeddings=embed_batch([query]), n_results=k, where={"turn": {"$lt": cutoff}})
    return sorted(res["documents"][0], key=lambda t: int(re.search(r"\[(\d+)턴", t).group(1)))

In [ ]:
# Ex31. GM 프롬프트에 [이전 기억] 넣기 + 매 턴 저장 (Ex11·Ex13 함수를 감싸요)
if "[이전 기억]" not in GM_TEMPLATE:
    GM_TEMPLATE = GM_TEMPLATE.replace("[최근 대화]", "[이전 기억] (지난 일이에요. 지금 장면처럼 쓰지 마세요)\n{memory}\n\n[최근 대화]")

RULE_MEM = ("7. 플레이어가 지난 일(누가 뭐라고 했는지 등)을 물으면, 새 장면을 묘사하지 말고 "
            "[이전 기억]과 [최근 대화]에 있는 내용으로 짧게 알려 주세요. 거기에 없으면 기억나지 않는다고 하세요.")
if "7. 플레이어가 지난 일" not in GM_TEMPLATE:
    GM_TEMPLATE = GM_TEMPLATE.replace("\n\n[현재 장소]", "\n" + RULE_MEM + "\n\n[현재 장소]")

def gm_prompt(action, state, hits, template=None):
    return (template or GM_TEMPLATE).format(
        room=rooms["rooms"][state["room"]]["name"],
        exits=", ".join(e["name"] for e in exits_of(state)) or "(없음)",
        docs="\n\n".join(f"- {h['text']}" for h in hits) or "(없음)",
        memory="\n".join(recall(action, state)) or "(없음)",
        history=fmt_history(state), action=action)

if not getattr(gm_turn, "has_memory", False):
    gm_turn_base = gm_turn

def gm_turn(action, state):
    r = gm_turn_base(action, state)
    if len(state["history"]) >= 2 and state["history"][-2][1] == action:
        remember(state, action, state["history"][-1][1])
    return r
gm_turn.has_memory = True

def new_game(role=ROLE):
    old = mem.get()["ids"]
    if old:
        mem.delete(ids=old)
    st = new_state(role)
    st["visited"], st["history"], st["turn"] = {st["room"]}, [], 0
    return st

---
## 13. 화면 (Gradio)
```
버튼/입력 → on_action → game_turn (이동·판정·전투·규칙 질문) → 채팅창 + 오른쪽 상태 패널
```
- Ex33: 상태 → 화면 글·버튼 (Gradio 없이 확인 가능)
- Ex34: Gradio 연결 후 실행. **마지막에 공개 링크가 나와요** (이 셀이 끝나기 전까지 화면이 켜져 있어요)
- 이 셀은 **맨 마지막에** 실행하세요 (위의 모든 셀이 먼저 실행돼 있어야 해요)

In [ ]:
# Ex33. 화면용 도우미
ROLE_KO = {"fighter": "전사", "rogue": "도적", "wizard": "마법사", "cleric": "성직자"}
MAX_EXITS = max(len(r["exits"]) for r in rooms["rooms"].values())

def chat_text(text):
    text = re.sub(r"\n*📍[^\n]*", "", text)
    return text.replace('("굴린다"라고 입력)', "(아래 🎲 버튼)").strip()

def panel_md(st):
    if st is None:
        return "### 상태\n직업을 고르고 **새 게임 시작**을 눌러 주세요."
    me, ch = COMBAT_STATS[st["role"]], CHARS[st["role"]]
    lines = [f"### 🛡️ {ROLE_KO[st['role']]}",
             f"❤️ HP **{st.get('hp', me['hp'])}/{me['hp']}** · AC {me['ac']} · 🗡️ {me['weapon']}",
             f"숙련: {', '.join(sorted(ch['skills']))}",
             "---",
             f"📍 **{rooms['rooms'][st['room']]['name']}**",
             f"장면 {st['scene']} · {st.get('turn', 0)}턴 · 가 본 방 {len(st['visited'])}곳"]
    c = st.get("combat")
    if c:
        lines += ["---", "⚔️ **전투 중**"] + [f"- 👹 {f['label']}: {f['hp']}/{f['max']}" for f in c["foes"] if f["hp"] > 0]
    else:
        ex = exits_of(st)
        if ex:
            lines += ["---", "**갈 수 있는 곳**"] + [f"- {e['name']}" for e in ex]
    return "\n".join(lines)

def dev_md(st, r):
    if st is None or r is None:
        return ""
    return "\n".join(["#### 🔧 개발자 정보",
                      f"- 종류: {r.get('kind', '-')} · {r.get('secs', 0):.1f}초",
                      f"- 근거: {(r.get('hits') or [])[:4]}",
                      f"- 검사: {r.get('problems') or '통과'}",
                      f"- 새 flag: {r.get('flags_new') or []}",
                      f"- 전체 flag: {sorted(st['flags'])}",
                      f"- room: `{st['room']}`"])

def buttons_of(st):
    if st is None or st.get("ended"):
        return {"exits": [], "dice": False, "start": False, "combat": False}
    if st.get("combat"):
        return {"exits": [], "dice": False, "start": False, "combat": True}
    return {"exits": [short(e["to"]) for e in exits_of(st)],
            "dice": bool(st.get("pending_check")),
            "start": bool(foe_docs_here(st)),
            "combat": False}

def run_turn(action, st):
    t0 = time.time()
    try:
        r = game_turn(action, st)
    except Exception as e:
        r = {"text": f"⚠️ 오류가 났어요: {type(e).__name__}: {e}\nOllama 서버가 켜져 있는지 확인해 주세요.", "hits": [], "problems": []}
    r["secs"] = time.time() - t0
    return r

def start_game(role):
    st = new_game(role)
    t0 = time.time()
    r = gm_turn("마을 광장에 도착해 주변을 둘러본다", st)
    st["pending_check"] = None
    r["secs"], r["kind"] = time.time() - t0, "진행"
    text = f"🔔 **잠긴 종** — {ROLE_KO[role]}의 모험이 시작돼요.\n\n" + re.sub(r"\n*🎲[^\n]*", "", chat_text(r["text"]))
    return st, text, r

In [ ]:
# Ex34. Gradio 화면
import inspect
import gradio as gr

CHAT_KW = {"type": "messages"} if "type" in inspect.signature(gr.Chatbot.__init__).parameters else {}
msg = lambda role, content: {"role": role, "content": content}
KEEP = gr.update()

def btn_updates(st):
    b = buttons_of(st)
    ups = [gr.update(value=f"🚪 {n}", visible=True) for n in b["exits"]]
    ups += [gr.update(visible=False)] * (MAX_EXITS - len(ups))
    return ups + [gr.update(visible=b["dice"]), gr.update(visible=b["start"])] + [gr.update(visible=b["combat"])] * 3

N_BTN = MAX_EXITS + 5

def pack(history, st, dev, box=KEEP, dev_text=""):
    return (history, box, st, panel_md(st), gr.update(value=dev_text, visible=dev), *btn_updates(st))

def busy(history, st):
    return (history, gr.update(value="", interactive=False), st, KEEP, KEEP, *[KEEP] * N_BTN)

def on_action(action, shown, history, st, dev):
    history = list(history or [])
    if st is None:
        yield pack(history + [msg("assistant", "먼저 위에서 직업을 고르고 **새 게임 시작**을 눌러 주세요.")], st, dev, gr.update(interactive=False))
        return
    if st.get("ended"):
        yield pack(history + [msg("assistant", "이야기가 끝났어요. **새 게임 시작**을 눌러 주세요.")], st, dev, gr.update(interactive=False))
        return
    m = re.match(r"이동\s*(\d+)$", action)
    if m and int(m.group(1)) <= len(exits_of(st)):
        shown = f"🚪 {short(exits_of(st)[int(m.group(1)) - 1]['to'])}"
    history += [msg("user", shown or action), msg("assistant", "⏳ GM이 생각하는 중…")]
    yield busy(history, st)
    r = run_turn(action, st)
    history[-1] = msg("assistant", chat_text(r["text"]))
    yield pack(history, st, dev, gr.update(value="", interactive=not st.get("ended")), dev_md(st, r))

def on_text(text, history, st, dev):
    text = (text or "").strip()
    if not text:
        yield (KEEP, KEEP, st, KEEP, KEEP, *[KEEP] * N_BTN)
        return
    yield from on_action(text, None, history, st, dev)

def on_start(role, dev):
    wait = [msg("assistant", "⏳ 모험을 준비하는 중… (첫 시작은 모델을 올리느라 1~2분 걸릴 수 있어요)")]
    yield (wait, gr.update(value="", interactive=False), None, KEEP, KEEP, *[KEEP] * N_BTN)
    try:
        st, text, r = start_game(role)
    except Exception as e:
        yield pack([msg("assistant", f"⚠️ 시작하지 못했어요: {type(e).__name__}: {e}\nOllama 서버와 모델(qwen2.5:14b)이 준비됐는지 확인해 주세요.")], None, dev, gr.update(interactive=False))
        return
    yield pack([msg("assistant", text)], st, dev, gr.update(value="", interactive=True), dev_md(st, r))

with gr.Blocks(title="잠긴 종 · TRPG GM", theme=gr.themes.Soft()) as demo:
    st_box = gr.State(None)
    gr.Markdown("# 🔔 잠긴 종 (The Drowned Bell)\n혼자 하는 D&D TRPG예요. AI가 GM이 되어 이야기를 이끌어요.")
    with gr.Row():
        role_dd = gr.Dropdown(choices=[(f"{ROLE_KO[r]} ({r})", r) for r in ROLES], value=ROLE, label="직업", scale=2)
        start_btn = gr.Button("🎲 새 게임 시작", variant="primary", scale=1)
        dev_cb = gr.Checkbox(label="개발자 모드", value=False, scale=1)
    with gr.Row():
        with gr.Column(scale=3):
            chat = gr.Chatbot(height=520, show_label=False, **CHAT_KW)
            with gr.Row():
                exit_btns = [gr.Button("", visible=False, size="sm") for _ in range(MAX_EXITS)]
            with gr.Row():
                dice_btn = gr.Button("🎲 굴린다", visible=False, variant="primary")
                fight_btn = gr.Button("⚔️ 전투 시작", visible=False, variant="stop")
                atk_btn = gr.Button("🗡️ 공격", visible=False, variant="stop")
                dodge_btn = gr.Button("🛡️ 회피", visible=False)
                flee_btn = gr.Button("🏃 도주", visible=False)
            box = gr.Textbox(placeholder="무엇을 할까요?  (규칙이 궁금하면 '?'로 시작: ? 이점이 뭐야)", show_label=False, interactive=False)
        with gr.Column(scale=1):
            panel = gr.Markdown(panel_md(None))
            dev_box = gr.Markdown(visible=False)

    OUTS = [chat, box, st_box, panel, dev_box, *exit_btns, dice_btn, fight_btn, atk_btn, dodge_btn, flee_btn]
    assert len(OUTS) == 5 + N_BTN
    ONCE = dict(concurrency_id="turn", concurrency_limit=1)

    start_btn.click(on_start, [role_dd, dev_cb], OUTS, **ONCE)
    box.submit(on_text, [box, chat, st_box, dev_cb], OUTS, **ONCE)

    def bind(btn, action, shown=None):
        def handler(h, s, d):
            yield from on_action(action, shown, h, s, d)
        btn.click(handler, [chat, st_box, dev_cb], OUTS, **ONCE)
    for i, b in enumerate(exit_btns):
        bind(b, f"이동 {i + 1}")
    bind(dice_btn, "굴린다", "🎲 굴린다")
    bind(fight_btn, "/전투", "⚔️ 전투 시작")
    bind(atk_btn, "1", "🗡️ 공격")
    bind(dodge_btn, "2", "🛡️ 회피")
    bind(flee_btn, "3", "🏃 도주")
    dev_cb.change(lambda v: gr.update(visible=v), dev_cb, dev_box)

demo.queue().launch(share=True, debug=True)

---
# 실험 · 확인


In [ ]:
# # Ex01. 확인
# print("문서:", len(docs), "| 방:", len(rooms["rooms"]), "| 시작 방:", rooms["start"])
# print("종류:", dict(Counter(d["meta"]["kind"] for d in docs)))
# print("장면별:", dict(sorted(Counter(d["meta"]["scene_id"] for d in docs).items())))
# print("\n예시:", docs[0]["id"], docs[0]["meta"])
# print(docs[0]["text"])

In [ ]:
# # Ex04. 확인
# state = new_state()
# print("시작 상태:", state)
# print("지금 볼 수 있는 문서:", sum(visible(d["meta"], state) for d in docs), "/", len(docs))

In [ ]:
# # Ex05. 확인
# def show(hits):
#     if not hits:
#         print("  (볼 수 있는 문서가 없어요)")
#     for h in hits:
#         print(f"  {h['score']:.3f} [{h['kind']}] {h['id']} | {h['text'][:70]}")

# show(search_scenario("우물을 살펴본다", new_state()))

## 6. 필터가 일하는지 눈으로 확인

In [ ]:
# # Ex06. 같은 질문인데 상태만 다르게
# q = "할빅이 숨기는 게 뭐지?"
# for scene in (2, 5):
#     st = {**new_state("fighter"), "scene": scene}
#     print(f"\n장면 {scene} · fighter · 방 조건 없음 · \"{q}\"")
#     show([h for h in search_scenario(q, st, k=3, use_room=False)])
#     print("  → scn-s1-secret-halvik 있음:", any(h["id"] == "scn-s1-secret-halvik"
#                                                for h in search_scenario(q, st, k=10, use_room=False)))

# q = "벽난로 위 석판의 문장은?"
# for role in ("cleric", "fighter"):
#     st = {**new_state(role), "room": "halvik_house"}
#     print(f"\n장면 1 · {role} · 촌장 집 · \"{q}\"")
#     show(search_scenario(q, st, k=3))

In [ ]:
# # Ex07. Chroma 필터 전수 확인
# leak = block = 0
# room_ids = list(rooms["rooms"])
# for scene in range(1, 7):
#     for role in ROLES:
#         for room in room_ids:
#             st = {"scene": scene, "role": role, "flags": set(), "room": room}
#             got = {h["id"] for h in search_scenario("아무 질문", st, k=len(docs))}
#             want = {d["id"] for d in docs if visible(d["meta"], st)}
#             leak += len(got - want)
#             block += len(want - got)
# print("조합:", 6 * len(ROLES) * len(room_ids), "| 누설:", leak, "| 과차단:", block)

## 7. 검색 확인 (질문 69개)
질문마다 (장면, 직업, flag, 방)이 정해져 있고, 정답 문서가 위 k개 안에 드는지 봐요.
질문은 시나리오를 쓰면서 함께 만든 것이라 **회귀 확인용**이에요 (성능 보고용 아님).

In [ ]:
# # Ex08. 정답이 몇 등에 나오나
# gold = json.load(open(GOLD_PATH, encoding="utf-8"))

# def rank_of(g, use_room):
#     st = {"scene": g["scene"], "role": g["role"], "flags": set(g["flags"]), "room": g["room"]}
#     got = [h["id"] for h in search_scenario(g["q"], st, k=10, use_room=use_room)]
#     return next((i + 1 for i, x in enumerate(got) if x in g["want"]), None), got

# for use_room in (False, True):
#     ranks, miss = [], []
#     for g in tqdm(gold, leave=False):
#         r, got = rank_of(g, use_room)
#         ranks.append(r)
#         if r is None or r > 3:
#             miss.append((g["q"][:30], g["want"][0], f"{r}등", got[:2]))
#     n = len(ranks)
#     c = [sum(1 for r in ranks if r and r <= k) for k in (1, 3, 5)]
#     print(f"방 조건 {'있음' if use_room else '없음'}: 1등 {c[0]}/{n} | 3등 안 {c[1]}/{n} | 5등 안 {c[2]}/{n}")
#     for m in miss:
#         print("   ", m)

In [ ]:
# # Ex10. 확인
# st = new_game()
# print("출구:", [(i + 1, e["name"]) for i, e in enumerate(exits_of(st))])
# print("'촌장 집으로 간다' →", detect_move("촌장 집으로 간다", st), "| '이동 2' →", detect_move("이동 2", st),
#       "| '우물을 본다' →", detect_move("우물을 본다", st))
# move(st, ["halvik_house"])
# print("촌장 집에서 '우물가로 간다' →", detect_move("우물가로 간다", st), "(가 본 적 없으면 None)")
# st["visited"].add("well_side")
# print("우물가에 가 봤다면 →", detect_move("우물가로 간다", st))

In [ ]:
# # Ex11. 확인
# st = new_game()
# print(gm_prompt("주변을 둘러본다", st, search_scenario("주변을 둘러본다", st))[:1200])

In [ ]:
# # Ex12. 확인
# print(split_check("문이 무겁게 버틴다.\n[판정] 근력(운동) DC 15"))
# print(gm_problems("HP 22의 슬라임이 있다"), gm_problems("탁한 물속에서 무언가가 천천히 움직인다. 물결이 일렁인다."))

In [ ]:
# # Ex14. 처음부터 몇 턴 해 보기 (dev=True면 근거 문서·상태도 보여요)
# !ollama list
# game = new_game(ROLE)
# for action in ["주변을 둘러본다", "우물을 자세히 살펴본다", "촌장 집으로 간다", "촌장에게 무슨 일인지 묻는다", "벽난로 위 석판을 본다"]:
#     play(action, dev=True)

In [ ]:
# # Ex15. 확인
# for t in ["근력(운동)", "지혜 (감지)", "도둑 도구", "설득"]:
#     a, s = parse_skill(t)
#     print(f"{t:10s} → {a}/{s} | 전사 {modifier('fighter', a, s):+d} | 도적 {modifier('rogue', a, s):+d}")

In [ ]:
# # Ex16. 확인
# r = roll_check(new_game("fighter"), "근력(운동)", 15)
# print(r)

In [ ]:
# # Ex17. 확인
# # 표가 데이터와 맞는지
# names = set(scenes["flags"])
# bad = [x for x in CHECK_FLAGS + EVENT_FLAGS if x[0] not in rooms["rooms"] or x[-1] not in names] + \
#       [f for f in COMBO_FLAGS if f not in names]
# print("표 오류:", bad)

In [ ]:
# # Ex19. 판정·flag 시험
# game = new_game("fighter")
# game["visited"] |= {"well_side", "shaft_upper", "ledge_60", "shaft_lower", "cloister_entry"}
# move(game, ["cloister_mid"])
# print("열리기 전 출구:", [e["name"] for e in exits_of(game)])
# play("서쪽 벽의 이끼 낀 문을 어깨로 부순다", dev=True)
# if game.get("pending_check"):
#     play("굴린다", dev=True)
# print("열린 뒤 출구:", [e["name"] for e in exits_of(game)], "| flag:", sorted(game["flags"]))

# game = new_game("fighter")
# play("세라에게 인형을 받는다", dev=True)

In [ ]:
# # Ex20. 확인
# print("룰 청크:", len(rr.chunks), "| 번역·검색 캐시:", len(rr._sf_cache), "| 모델:", rr.GEN_MODEL)

In [ ]:
# # Ex21. 확인
# for t in ["넘어진 상태에서 일어나려면 이동력이 얼마나 들어?", "우물 안을 들여다본다", "할빅은 뭘 숨기고 있어?", "규칙: 기회 공격"]:
#     print(classify(t), "|", t)

In [ ]:
# # Ex22. 분류 확인
# rule_qs = [g["q"] for g in rr.golden_ko + rr.golden_new]
# scn_qs = [g["q"] for g in json.load(open(GOLD_PATH, encoding="utf-8"))]
# wrong = []
# for want, qs in (("규칙", rule_qs), ("진행", scn_qs)):
#     ok = 0
#     for q in tqdm(qs, leave=False):
#         got = classify(q)
#         ok += got == want
#         if got != want:
#             wrong.append((want, got, q))
#     print(f"{want}: {ok}/{len(qs)}")
# for w in wrong:
#     print("  틀림", w)

In [ ]:
# # Ex24. 섞어서 해 보기
# game = new_game(ROLE)
# for action in ["주변을 둘러본다", "넘어진 상태에서 일어나려면 이동력이 얼마나 들어?", "우물가로 간다", "?독에 중독되면 뭐가 불리해져"]:
#     play(action, dev=True)

In [ ]:
# # Ex25. 확인
# for did, found in SCN_MONSTERS.items():
#     print(f"{did:28s} 방 {by_id[did]['meta']['room'] or '-':18s}", [(n, k, srd_monster(n)) for n, k in found] or "⚠️ 못 읽음")

In [ ]:
# # Ex25. 확인
# print(ko_monster("종실에 Shadow 3마리. 회랑 물속에 Gray Ooze가 있다."))

In [ ]:
# # Ex29. 전투 시험
# game = new_game("fighter")
# game["visited"] |= {"well_side", "shaft_upper", "ledge_60", "shaft_lower", "cloister_entry"}
# move(game, ["cloister_mid"])
# print("이 방의 적:", foe_docs_here(game))
# play("/전투", dev=True)
# for _ in range(8):
#     if not game.get("combat"):
#         break
#     play("1", dev=True)

In [ ]:
# # Ex32. 기억 시험
# game = new_game(ROLE)
# for action in ["촌장 집으로 간다", "할빅에게 실종자에 대해 묻는다", "우물 광장으로 간다", "대장간과 여관으로 간다",
#                "여관 주인에게 소문을 묻는다", "우물 광장으로 간다", "우물가로 간다", "할빅이 실종자에 대해 뭐라고 했었지?"]:
#     play(action, dev=True)
# print("\n[마지막 턴에 들어간 기억]")
# for m in recall("할빅이 실종자에 대해 뭐라고 했었지?", game):
#     print(" ", m[:120])

In [ ]:
# # 직접 해 보기
# play("우물 안을 들여다본다", dev=True)

In [ ]:
# # Ex33. 확인
# _t = new_state("fighter"); _t["visited"] = {_t["room"]}
# print(buttons_of(_t)); print(panel_md(_t))